In [3]:
from spikeinterface.sorters import installed_sorters
installed_sorters()

['kilosort4', 'lupin', 'simple', 'spykingcircus2', 'tridesclous2']

In [ ]:
import spikeinterface.full as si
import matplotlib.pyplot as plt
import numpy as np
import probeinterface as pi
from pathlib import Path
import pandas as pd 
import os, sys
import shutil
from pprint import pprint 
import time as time
import json
%load_ext autoreload
%autoreload 2

import bombcell as bc

    
global_job_kwargs = dict(n_jobs=-3, chunk_duration="10s",progress_bar=True)
si.set_global_job_kwargs(**global_job_kwargs)


basefolder= r"H:\Data\raw\8186_naive"
base_path = Path(basefolder)



recording =  si.read_spikeglx(basefolder, stream_id='imec0.ap', load_sync_channel=False)
lfp = si.read_spikeglx(basefolder, stream_id='imec0.lf', load_sync_channel=False)
event =  si.read_spikeglx(basefolder, stream_id='nidq', load_sync_channel=False)
print(recording)

metapath = base_path / 'Meta'
if not os.path.isdir(metapath):
   os.makedirs(metapath)


In [ ]:
rec1 = si.bandpass_filter(recording)
rec1 = si.phase_shift(rec1)
bad_channel_ids, channel_labels = si.detect_bad_channels(rec1,method = 'coherence+psd')
print(bad_channel_ids)
rec1 = si.interpolate_bad_channels(recording=rec1, bad_channel_ids=bad_channel_ids)

rec1 = si.common_reference(rec1, operator="median", reference="global")
print(rec1)


%matplotlib widget
si.plot_traces({'raw':recording,'filtered':rec1}, backend='ipywidgets')

from spikeinterface.sorters import installed_sorters
installed_sorters()
import torch
print(torch.cuda.is_available())
print(torch.cuda.current_device())
torch.cuda.get_device_name(0)

In [ ]:
Sorting_KS4 = si.run_sorter(sorter_name="kilosort4", recording=rec1, folder=basefolder + str('/sorted'),remove_existing_folder=True)
analyzer = si.create_sorting_analyzer(Sorting_KS4, rec1, sparse=True, format='binary_folder',folder=basefolder +str('/analyzer'))

analyzer.compute(['random_spikes', 'waveforms', 'templates', 'noise_levels','unit_locations','correlograms'],**global_job_kwargs)
analyzer.compute('spike_amplitudes')
analyzer.compute('principal_components', n_components = 5, mode="by_channel_local",**global_job_kwargs)
analyzer.compute(["spike_locations", "template_metrics", "quality_metrics"],**global_job_kwargs)

plot_path =base_path/'plots'
spike_times_path = base_path / 'spike_times'


# Load Data

sf = recording.sampling_frequency

# 1. Bombcell & Labels
bc_results = si.bombcell_label_units(analyzer, thresholds=si.bombcell_get_default_thresholds())
labels = bc_results["bombcell_label"].values
analyzer.sorting.set_property("bombcell_label", labels)

# Save Labels Plot
w = si.plot_unit_labels(analyzer, labels, ylims=(-300, 100))
w.figure.savefig(plot_path / "bombcell_labels.png")
plt.close(w.figure)



spikes = pd.DataFrame(analyzer.sorting.to_spike_vector())
label_map = pd.Series(labels, index=range(len(analyzer.unit_ids)))
spikes['label'] = spikes['unit_index'].map(label_map)
spikes['time_s'] = spikes['sample_index'] / analyzer.sampling_frequency
spike_df = spikes.drop(columns=['segment_index'], errors='ignore')
spike_df.to_csv(spike_times_path / 'spike_times.csv', index=False)


# Convert to records and save
structured_array = spike_df.to_records(index=False)


np.save(spike_times_path / 'spike_times.npy', structured_array)



    
    # Convert to records and save
    structured_array = spike_df.to_records(index=False)
    
    
    np.save(spike_times_path / 'spike_times.npy', structured_array)

In [ ]:

from scipy.signal import medfilt
from scipy.interpolate import interp1d


def ttl_from_analog(signal, fs, hysteresis=0.1, filt_kernel=5):
    """
    Analog to digital TTL conversion with hysteresis.
    Uses proper forward-fill to maintain binary state between transitions.
    """
    signal_f = medfilt(signal, kernel_size=filt_kernel)

    v_low  = np.percentile(signal_f, 5)
    v_high = np.percentile(signal_f, 95)
    v_lo   = v_low  + hysteresis * (v_high - v_low)
    v_hi   = v_high - hysteresis * (v_high - v_low)

    above_hi = signal_f > v_hi
    below_lo = signal_f < v_lo

    hi_crossings = np.where(np.diff(above_hi.astype(np.int8)) ==  1)[0] + 1
    lo_crossings = np.where(np.diff(below_lo.astype(np.int8)) ==  1)[0] + 1

    # Sparse change array, forward-filled via index propagation
    changes = np.full(len(signal_f), np.nan)
    changes[0]            = 1.0 if signal_f[0] > v_hi else 0.0
    changes[hi_crossings] = 1.0
    changes[lo_crossings] = 0.0

    mask      = ~np.isnan(changes)
    last_seen = np.where(mask, np.arange(len(changes)), 0)
    np.maximum.accumulate(last_seen, out=last_seen)
    digital   = changes[last_seen].astype(np.int8)

    d           = np.diff(digital)
    rising_idx  = np.where(d ==  1)[0] + 1
    falling_idx = np.where(d == -1)[0] + 1

    return digital, rising_idx, falling_idx


def quadrature_speed_direction(sigA, sigB, fs, pulses_per_rev=900,
                               hysteresis=0.1, max_gap_s=0.020):
    """
    x4 quadrature decoder.

    Uses all transitions on both channels (4 x pulses_per_rev events/rev).
    Inserts zero-speed sentinels exactly max_gap_s after each long inter-edge
    gap, so stopped periods and brief movements both resolve correctly.

    Parameters
    ----------
    sigA, sigB      : raw analog encoder channels
    fs              : sampling frequency (Hz)
    pulses_per_rev  : encoder lines per revolution (900 for H5-900-NE-S)
    hysteresis      : fraction of signal range used as hysteresis band
    max_gap_s       : inter-edge gap (s) above which speed is set to 0;
                      ~20 ms corresponds to ~1 cm/s at the mouse's position
                      on a 6 cm radius wheel
    """
    sig_len   = len(sigA)
    time_full = np.arange(sig_len) / fs

    digA, rising_A, falling_A = ttl_from_analog(sigA, fs, hysteresis)
    digB, rising_B, falling_B = ttl_from_analog(sigB, fs, hysteresis)

    # --- x4 decoding: direction logic for each edge type ---
    #   A rises  : forward if B=0, backward if B=1
    #   A falls  : forward if B=1, backward if B=0
    #   B rises  : forward if A=1, backward if A=0
    #   B falls  : forward if A=0, backward if A=1
    edges = np.concatenate([rising_A,  falling_A,
                             rising_B,  falling_B])
    dirs  = np.concatenate([
        np.where(digB[rising_A]  == 0,  1, -1),
        np.where(digB[falling_A] == 1,  1, -1),
        np.where(digA[rising_B]  == 1,  1, -1),
        np.where(digA[falling_B] == 0,  1, -1),
    ])

    if len(edges) < 2:
        return time_full * 1000, np.zeros(sig_len), np.zeros(sig_len)

    sort_idx = np.argsort(edges, kind='stable')
    edges    = edges[sort_idx]
    dirs     = dirs[sort_idx]

    t_edges      = edges / fs
    deg_per_edge = 360.0 / (pulses_per_rev * 4)

    dt          = np.diff(t_edges)
    speed_edges = (deg_per_edge * dirs[:-1]) / dt  # direction at interval start
    t_mid       = (t_edges[1:] + t_edges[:-1]) / 2

    # --- Zero-speed sentinels ---
    # Placed exactly max_gap_s after the last edge before each long gap,
    # so brief fast movements drop to zero promptly rather than at gap midpoint.
    long_gaps   = np.where(dt > max_gap_s)[0]
    t_zeros     = t_edges[long_gaps] + max_gap_s
    speed_zeros = np.zeros(len(long_gaps))
    dir_zeros   = np.zeros(len(long_gaps))

    t_mid_aug = np.concatenate([t_mid,       t_zeros])
    speed_aug = np.concatenate([speed_edges, speed_zeros])
    dir_aug   = np.concatenate([dirs[:-1],   dir_zeros])

    sort      = np.argsort(t_mid_aug, kind='stable')
    t_mid_aug = t_mid_aug[sort]
    speed_aug = speed_aug[sort]
    dir_aug   = dir_aug[sort]

    f_speed = interp1d(t_mid_aug, speed_aug, kind='previous',
                       bounds_error=False, fill_value=0)
    f_dir   = interp1d(t_mid_aug, dir_aug,   kind='previous',
                       bounds_error=False, fill_value=0)

    return time_full * 1000, f_speed(time_full), f_dir(time_full)


def extract_ttl_from_bit(digital_word, bit, sampling_rate, plot_path, savename, plot=True):
    ttl_signal = (digital_word >> bit) & 1
    time_axis = np.arange(len(ttl_signal)) / sampling_rate
    if plot:
        plt.figure(figsize=(15, 3))
        plt.plot(time_axis, ttl_signal)
        plt.title(f'Isolated Bit {bit} ({savename})')
        plt.savefig(plot_path / f"ttl_bit_{bit}_{savename}.png")
        plt.close()
    diff = np.diff(ttl_signal)
    rising_idx = np.where(diff > 0)[0]
    falling_idx = np.where(diff < 0)[0]
    edges = np.concatenate([rising_idx / sampling_rate, falling_idx / sampling_rate])
    types = (['rising'] * len(rising_idx)) + (['falling'] * len(falling_idx))
    return pd.DataFrame({'timestamps': edges, 'edge_type': types}).sort_values('timestamps').reset_index(drop=True)

def extract_and_save_ttl_events(data, bit_name_pairs, save_path, plot_path):
    # CRITICAL: return_scaled=False so bitwise >> works on integers
    digital_signals = data.get_traces(return_in_uV=False)
    digital_word = digital_signals[:, 8]
    sampling_rate = data.get_sampling_frequency()
    for bit, savename in bit_name_pairs:
        ttl_df = extract_ttl_from_bit(digital_word, bit, sampling_rate, plot_path, savename)
        ttl_df.to_csv(save_path / f"{savename}.csv", index=False)
        
 # 4. Camera & Audio/State TTLs
cam_signal = event.get_traces(channel_ids=[event.get_channel_ids()[3]]).squeeze()
_, _, falling_A = ttl_from_analog(cam_signal, sf)
if len(falling_A) > 0:
    pd.DataFrame({'camttl': [falling_A[0] / sf]}).to_csv(metapath / 'camttl.csv', index=False)

pairs = [(1, 'State_changes'), (3, 'Audio')]
extract_and_save_ttl_events(event, pairs, metapath, plot_path)


#speed

 try:
        event = si.read_spikeglx(str(basefolder), stream_id='nidq',
                                  load_sync_channel=False)
        sf = event.get_sampling_frequency()

        sigA = event.get_traces(channel_ids=[event.get_channel_ids()[6]]).squeeze()
        sigB = event.get_traces(channel_ids=[event.get_channel_ids()[5]]).squeeze()

        ts_ms, speed, direction = quadrature_speed_direction(
            sigA, sigB, sf,
            pulses_per_rev=900,
            hysteresis=0.1,
            max_gap_s=0.020
        )

        df = pd.DataFrame({
            'time_ms'  : ts_ms,
            'speed'    : speed,
            'direction': direction
        })
        save_path = meta_path / 'speed.csv'
        df.to_csv(save_path, index=False)
        print(f"  Saved CSV : {save_path}")

        # --- Plot: stride-decimate to preserve spike shapes ---
        num_downsample_points = 5000
        if len(speed) > num_downsample_points:
            stride    = len(speed) // num_downsample_points
            speed_ds  = speed[::stride]
            time_s_ds = ts_ms[::stride] / 1000
        else:
            speed_ds  = speed
            time_s_ds = ts_ms / 1000

        plt.figure(figsize=(10, 4))
        plt.plot(time_s_ds, speed_ds, color='crimson', linewidth=1.5, alpha=0.8)
        plt.title(f"Velocity Over Time — {basefolder.name}", fontsize=12, pad=10)
        plt.xlabel("Time (s)", fontsize=10)
        plt.ylabel("Speed (deg/s)", fontsize=10)
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout()

        plot_save_path = plots_path / 'speed_overview.png'
        plt.savefig(plot_save_path, dpi=200)
        plt.close()
        print(f"  Saved plot: {plot_save_path}")